# 三接收器校准

## 简介

人们早已知道，只要 VNA 只有三个接收器且没有内部同轴开关，就可以进行完全误差校正。然而，由于现代 VNA 都不采用这种架构，在当今的现代 VNA 上无法获得进行完全校正测量的软件。

最近，只包含三个接收器的频率扩展单元的应用变得更加普遍。此外，低成本的 VNA 正成为学习 RF 电子技术的有用工具。代表性的例子是 NanoVNA，它使用三接收器设计。因此，对于具有三个接收器且没有内部同轴开关的系统，需要进行完全误差校正能力。本文档介绍如何使用 [scikit-rf](http://www.scikit-rf.org) 来完全校正在该系统上进行的双端口测量。

## 理论

下图显示了无开关三接收器系统的电路模型。

In [ ]:
from IPython.display import SVG, Image

SVG('three_receiver_cal/pics/vnaBlockDiagramForwardRotated.svg')

要完全校正任意双端口设备，必须以两种取向（称为正向和反向）测量设备。由于不存在开关，这需要操作员物理翻转设备，并保存这对测量数据。在晶圆上测量时，可以制造两个相同的设备，一个一个取向。无论哪种情况，在进行校正之前，每个 DUT 都需要一对测量数据。

虽然实际上设备正在翻转，但可以想象设备是静止的，而整个 VNA 电路翻转。这种解释适合实现，因为只需将正向误差系数复制到对应的反向误差系数中，即可重用现有的 12 项校正。这就是 `scikit-rf` 内部所做的事情。


## 示例演示

此示例演示如何使用 Agilent PNAX 和一套 VDI WR-12 TXRX-RX 频率扩展头获取的测量数据，从 [TwoPortOnePath](../../api/calibration/generated/skrf.calibration.calibration.TwoPortOnePath.rst) 和 [EnhancedResponse](../../api/calibration/generated/skrf.calibration.calibration.EnhancedResponse.rst) 校准。通过校正不对称的 DUT 来比较这两种算法。

### 测量校准标准和被测设备

首先，按照标准的 SOLT 校准程序在 VNA 上测量 Open、Short 和 Load 标准的 S 参数。我们假设读者熟悉 scikit-rf 中的此任务。如果不熟悉，在 [SOLT 示例](./SOLT.ipynb)中有详细描述。

校准三接收器 VNA 时，过程几乎完全相同。但请注意有几个细微差别。首先，Open、Short 和 Load 标准只在端口 1 上测量，而不是端口 2。VNA 无法反向测量，因此没有理由在端口 2 上进行 OSL 校准（除非需要隔离校准）。

尽管如此，scikit-rf 仍然期望输入双端口网络，因此我们仍然将所有结果保存为双端口网络。在这种情况下，只有 $S_{11}$ 和 $S_{21}$ 是实际测量值，$S_{12}$ 和 $S_{22}$ 未使用、没有意义，可以包含任意数据。可以使用全零值作为占位符。在计算目的上，scikit-rf 内部将设置 $S_{22} = S_{11}$ 和 $S_{12} = S_{21}$。

然后，DUT 以正向方向作为双端口网络测量，物理翻转，然后在反向取向再次测量。

### 读取数据

校准标准和 DUT 的测量数据通过将原始 s 参数数据保存为 touchstone 文件到磁盘，从 VNA 下载。

In [ ]:
! ls three_receiver_cal/data/

scikit-rf 可以使用以下方法将这些文件读取为 `Network`。

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf

%matplotlib inline

rf.stylely()


raw = rf.io.general.read_all_networks('three_receiver_cal/data/')
# list the raw measurements
raw.keys()

每个 `Network` 都可以通过字典 `raw` 访问。

In [ ]:
thru = raw['thru']
thru

如果查看 flush thru 的*原始*测量，可以看到只有 $S_{11}$ 和 $S_{21}$ 包含有意义的数据。其他 s 参数是噪声。

In [ ]:
thru.plot_s_db()

### 创建校准

在下面的代码中，从校准标准的相应测量响应和理想响应创建 TwoPortOnePath 校准。测量网络从磁盘读取，而它们对应的理想响应使用 scikit-rf 生成。有关使用 scikit-rf 进行离线校准的更多信息，可以在 [这里](../../tutorials/Calibration.ipynb)找到。

In [ ]:

from skrf.calibration import TwoPortOnePath
from skrf.constants import mil
from skrf.media import RectangularWaveguide
from skrf.network import two_port_reflect as tpr

# pull frequency information from measurements
frequency = raw['short'].frequency

# the media object
wg = RectangularWaveguide(frequency=frequency, a=120*mil, z0_override=50)

# list of 'ideal' responses of the calibration standards
ideals = [wg.short(nports=2),
          tpr(wg.delay_short( 90,'deg'), wg.match()),
          wg.match(nports=2),
          wg.thru()]

# corresponding measurements to the 'ideals'
measured = [raw['short'],
            raw['quarter wave delay short'],
            raw['load'],
            raw['thru']]

# the Calibration object
cal = TwoPortOnePath(measured = measured, ideals = ideals)

默认情况下，`TwoPortOnePath` 假设所有校准和测量中端口 1 都是主动源（因此只有 $S_{11}$ 和 $S_{21}$ 是有效数据）。如果由于某种原因，有效测量值以 $S_{12}$ 和 $S_{22}$ 的形式呈现，可以在创建 `TwoPortOnePath` 对象时设置可选参数 `source_port=2`。

### 应用校正

三接收器系统可以进行两种类型的校正。

1. 完全校正
2. 部分校正

`scikit-rf` 对两者使用相同的 `Calibration` 对象，但根据 DUT 的 `type` 采用不同的校正算法。此示例中使用的 DUT 是 WR-15 衬垫与 WR-12 1" 直波导的级联，如下图所示。对 DUT 的测量使用*完全*校正和*部分*校正进行校正，并在下面比较结果。

In [ ]:
Image('three_receiver_cal/pics/asymmetic DUT.jpg', width='75%')

#### 完全校正

完全校正是通过在正向和反向两种取向下测量每个设备来实现的。需要明确的是，这意味着必须物理移除 DUT、翻转并重新插入。然后将这对测量数据传递给 `apply_cal()` 函数作为元组。这返回单个校正后的响应。


#### 部分校正

如果将单个测量传递给 `apply_cal()` 函数，则校准将采用部分校正。这种类型的校正称为 `EnhancedResponse`。根据测量应用，这种类型的校正可能*足够好*，甚至可能是唯一的选择。

### 对比

下面是对使用*完全*和*部分*算法校正的上图所示的 DUT 的直接比较。它显示部分校准在反射测量上产生较大的波纹，在透射测量上产生稍大的波纹。

In [ ]:
simulation = raw['simulation']

dutf = raw['wr15 shim and swg (forward)']
dutr = raw['wr15 shim and swg (reverse)']

corrected_full =     cal.apply_cal((dutf, dutr))
corrected_partial =  cal.apply_cal(dutf)



# plot results

f, ax = plt.subplots(1,2, figsize=(8,4))

ax[0].set_title ('$S_{11}$')
ax[1].set_title ('$S_{21}$')

corrected_partial.plot_s_db(0,0, label='Partial Correction',ax=ax[0])
corrected_partial.plot_s_db(1,0, label='Partial Correction',ax=ax[1])

corrected_full.plot_s_db(0,0, label='Full Correction', ax = ax[0])
corrected_full.plot_s_db(1,0, label='Full Correction', ax = ax[1])

simulation.plot_s_db(0,0,label='Simulation', ax=ax[0], color='k')
simulation.plot_s_db(1,0,label='Simulation', ax=ax[1], color='k')

plt.tight_layout()

### 如果我的被测设备是对称的怎么办？

如果已知 DUT 是**互易的**（ $S_{21}=S_{12}$ ）和**对称的**（ $S_{11}=S_{22}$ ），那么它在正向和反向取向下的响应应该是相同的。在这种情况下，不需要测量设备两次，可以避免。这在示例中进行了探讨：[TwoPortOnePath, EnhancedResponse, and FakeFlip](TwoPortOnePath, EnhancedResponse, and FakeFlip.ipynb)